# SuperCompress + AgentOps

Compress bulky agent context before the LLM call, then observe token/cost impact in AgentOps.

- [SuperCompress](https://www.supercompress.dev)
- [AgentOps](https://agentops.ai)


In [ ]:
%pip install -U agentops supercompress openai python-dotenv


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["AGENTOPS_API_KEY"] = os.getenv("AGENTOPS_API_KEY", "your_api_key_here")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your_openai_api_key_here")
os.environ.setdefault("SUPERCOMPRESS_API_KEY", os.getenv("SUPERCOMPRESS_API_KEY", ""))

import agentops
from openai import OpenAI
from supercompress import compress_for_turn


In [ ]:
BULKY_CONTEXT = """
Root cause: null deref in checkout when cart.line_items is empty.
Fix hint: guard cart.line_items before mapping SKUs.
Passed unit tests for shipping calculator.
Noise: redis reconnect logs repeating 40 times.
Noise: webpack rebuild spam.
GraphQL schema noise...
"""
USER_ASK = "Why does checkout crash on empty cart?"

agentops.init(auto_start_session=False, trace_name="SuperCompress Example")
tracer = agentops.start_trace(
    trace_name="SuperCompress Example",
    tags=["supercompress-example", "agentops-example", "token-savings"],
)

result = compress_for_turn(
    context="",
    user_query=USER_ASK,
    context_blocks=[BULKY_CONTEXT],
    budget_ratio=0.35,
    mode="compiler",
)
print(f"{result.original_tokens} → {result.kept_tokens} tokens ({result.tokens_saved_pct}% saved)")

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": result.compressed_text},
        {"role": "user", "content": USER_ASK},
    ],
)
print(response.choices[0].message.content)

agentops.end_trace(tracer, end_state="Success")
agentops.validate_trace_spans(trace_context=tracer)
print("✅ AgentOps spans OK")
